# Task 4
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [9]:
import pandas as pd
import numpy as np
from tqdm import tqdm
#from .autonotebook import tqdm as notebook_tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoConfig, AutoModel
from transformers import AlbertTokenizer, AlbertModel

from statistics import mean 

# Add any other imports you need here

Depending on your approach, you might need to adapt the structure of this template or parts not marked by TODOs.
It is not necessary to completely follow this template. Feel free to add more code and delete any parts that are not required.

In [10]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64  # TODO: Set the batch size according to both training performance and available memory
NUM_EPOCHS = None  # TODO: Set the number of epochs
NUM_WORKERS = 4

train_val = pd.read_csv("train.csv")
test_val = pd.read_csv("test_no_score.csv")

In [11]:
# TODO: Fill out ReviewDataset
class ReviewDataset(Dataset):
    def __init__(self, data_frame, is_train):
        self.data = data_frame
        self.is_train = is_train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        data = self.data.iloc[index]
        text = data['title'] +': '+ data['sentence'] #just concatenates
        if self.is_train:
            score = data['score']
            return text, torch.tensor(score, dtype=torch.float)
        else:
            return text

In [12]:
train_val = pd.read_csv("train.csv")
test_val = pd.read_csv("test_no_score.csv")

train_dataset = ReviewDataset(train_val,True)
print("Train Dataset 0:",train_dataset[0])
test_dataset = ReviewDataset(test_val, False)
#print("Test Dataset 0:",test_dataset[0])

Train Dataset 0: ('not really sayin anything: this movie is ok but its not teachin anything it doesnt tell you sny history really and it just get borning in some parts but it ok to watch if you wont to know some of the problems people fce daily', tensor(4.6579))


In [13]:
num_reviews = len(train_dataset)
train_loader = DataLoader(dataset=train_dataset, batch_size = BATCH_SIZE, shuffle=False, num_workers = 0, pin_memory = True)
print("Train Loader done:", len(train_loader.dataset))
test_loader = DataLoader(dataset=test_dataset, batch_size = BATCH_SIZE, shuffle=False, num_workers = 0, pin_memory = True)
print("Test Loader done:", len(test_loader.dataset))

Train Loader done: 12500
Test Loader done: 1000


In [14]:
def getEmbeddings(model, tokenizer, loader, train=True): 
    embeddings = []
    labels = []
    c = 0
    if train:
        print(model)
    model.eval()
    with torch.no_grad():
        for batch in loader:
            if train: 
                texts, scores = batch
            else: 
                texts = batch
            #print(type(texts))
            inputs = tokenizer(texts, padding='max_length', truncation=True, return_tensors='pt', max_length=512)
            inputs = inputs.to(DEVICE)
            embeds = model(**inputs).last_hidden_state[:,0,:]
            #print(embeds.shape)
            embeddings.append(embeds.cpu())

            """
            #different way of embedding, seems worse
            inputs = tokenizer.batch_encode_plus(texts, padding='max_length', truncation=True, return_tensors='pt', max_length=512)
            inputs = inputs.to(DEVICE)
            embeds = model(**inputs).pooler_output
            #print(embeds.shape)
            embeddings.append(embeds)
            """

            #to avoid running out of memory
            del inputs, embeds
            torch.cuda.empty_cache()

            c += BATCH_SIZE
            if(c % 512 == 0):  
                print(c)
            if train:
                labels.append(scores.cpu().numpy())
                
    print(len(embeddings),embeddings[0].shape)
    embeddings = torch.vstack(embeddings).numpy()
    print(embeddings.shape)
    if train:
        labels = np.concatenate(labels)
        np.save('train_embeddings1.npy', embeddings)
        np.save('train_labels1.npy',labels)
    else: 
        np.save('test_embeddings1.npy', embeddings)

In [15]:
#Attempt #1 at embeddings - seems to work better
"""
tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')
tokenizer.padding_side = "right" # says this is recommended on hugging faces
tokenizer.pad_token = tokenizer.eos_token 
model = AlbertModel.from_pretrained('albert/albert-base-v2')
"""

#tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

#another way to do it
tokenizer = AutoTokenizer.from_pretrained('albert/albert-base-v2')
model = AutoModel.from_pretrained('albert/albert-base-v2')

""" how they did it in the Q&A but doesn't work
tokenizer = AutoTokenizer.from_pretrained('albert/albert-base-v2')
config = AutoConfig.from_pretrained('albert/albert-base-v2')
model = AutoModel.from_config(config)
"""

model = model.to(DEVICE)

C:\Users\elena\anaconda3\envs\introtoml\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
#only has to be run once
getEmbeddings(model,tokenizer,train_loader,True)
print("train embeddings done")

getEmbeddings(model,tokenizer,test_loader,False)
print("test embeddings done")

AlbertModel(
  (embeddings): AlbertEmbeddings(
    (word_embeddings): Embedding(30000, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0, inplace=False)
  )
  (encoder): AlbertTransformer(
    (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
    (albert_layer_groups): ModuleList(
      (0): AlbertLayerGroup(
        (albert_layers): ModuleList(
          (0): AlbertLayer(
            (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (attention): AlbertAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (attention_dropout): Dropout(p=0, inplace=False)
      

In [16]:
#load embeddings
#train_embeddings1: Albert with AutoModel
#train_embeddings: Albert with AlbertModel
train_embeddings= np.load('train_embeddings1.npy')
train_labels = np.load('train_labels1.npy')
test_embeddings = np.load('test_embeddings1.npy')

#normalize embeddings maybe 
# doesn't really affect much it seems like, so let's not as we might want the directions or something from the embedding
"""train_embeddings = train_embeddings / (np.linalg.norm(train_embeddings,axis =1).reshape(-1,1)) #just divide by norm
test_embeddings = test_embeddings / (np.linalg.norm(test_embeddings,axis =1).reshape(-1,1))"""


print(train_embeddings.shape)
print(train_labels.shape)
#print(train_labels[0:5])
print(test_embeddings.shape)

(12500, 768)
(12500,)
(1000, 768)


In [17]:
#new datasets from embeddings
TRAINING_BATCH_SIZE = 64
TESTING_BATCH_SIZE = 64
new_training_dataset =  TensorDataset(torch.from_numpy(train_embeddings).type(torch.float), torch.from_numpy(train_labels).type(torch.float))
new_testing_dataset = TensorDataset(torch.from_numpy(test_embeddings).type(torch.float))
entire_training_loader = DataLoader(new_training_dataset,batch_size = TRAINING_BATCH_SIZE,pin_memory=True, num_workers=NUM_WORKERS,shuffle=True)
print(new_training_dataset[0:5])
#test loader
testing_loader = DataLoader(new_testing_dataset, batch_size = TESTING_BATCH_SIZE,pin_memory=True, num_workers=NUM_WORKERS,shuffle=False)

#Train-validation split
train_dataset, valid_dataset = torch.utils.data.random_split(new_training_dataset, [0.8,0.2])
training_loader = DataLoader(train_dataset,batch_size = TRAINING_BATCH_SIZE,pin_memory=True, num_workers=NUM_WORKERS,shuffle=True)
validation_loader = DataLoader(valid_dataset,batch_size = TESTING_BATCH_SIZE,pin_memory=True, num_workers=NUM_WORKERS,shuffle=True)
print("Entire Training data length:",len(entire_training_loader.dataset))
print("Training data length:",len(training_loader.dataset))
print("Validation data length:",len(validation_loader.dataset))

(tensor([[ 1.7308,  0.0319,  2.5344,  ..., -1.3469, -0.6268,  1.0349],
        [ 0.4305,  1.9400,  0.1163,  ..., -2.1513, -1.1883,  2.2192],
        [ 0.6939,  1.4691,  0.8903,  ..., -1.8379, -1.1285,  1.3993],
        [ 0.4249,  1.1430, -0.9427,  ..., -2.0752, -0.8130,  1.8368],
        [ 1.2784,  0.8008, -0.8079,  ..., -1.8521, -2.0881,  2.0256]]), tensor([4.6579, 7.5313, 9.5848, 1.0164, 9.6057]))
Entire Training data length: 12500
Training data length: 10000
Validation data length: 2500


#### Notes
- adding an extra layer doesn't seem to improve
- dropout worsens performance apparently
- ReLU seems to work better than SiLU and GELU or at least not significantly better

In [18]:
# TODO: Fill out MyModule
class MyModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(768,256),
            #nn.Dropout(0.7), #Dropout doesn't seem necessary & makes performance worse
            nn.BatchNorm1d(256), 
            nn.ReLU(),
            nn.Linear(256,64),
            #nn.Dropout(0.5), 
            nn.BatchNorm1d(64), 
            nn.ReLU(),  
            nn.Linear(64,1))
    def forward(self, x):
       return self.layers(x)


In [30]:
# TODO: Setup loss function, optimiser, and scheduler
model = MyModule().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001) #B1 = 0.9, β₂ = 0.999, ε = 1e-8.
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.01) 

"""#DEBUGGING
test,rest= torch.utils.data.random_split(new_training_dataset, [0.001,0.999])
test_loader = DataLoader(test,batch_size = 32,pin_memory=True, num_workers=NUM_WORKERS,shuffle=True)
print(len(test_loader.dataset))"""           

print(model)
for epoch in range(500):
    #Training loop
    train_losses = []
    model.train()
    for x_batch, y_batch in training_loader:
        x_batch, y_batch = x_batch.to(DEVICE),y_batch.to(DEVICE).unsqueeze(1)
        #y_batch = y_batch.unsqueeze(1).detach()
        #print(x_batch.shape,y_batch.shape)
        y_pred = model(x_batch)
        #print(y_pred)
        loss = criterion(y_pred, y_batch)
        train_losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    #print(train_losses)
    print(f'epoch: {epoch:2}  Training loss: {mean(train_losses)}')

    #Validation loop
    val_losses = []
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in validation_loader: 
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE).unsqueeze(1)
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            val_losses.append(loss.item())
            #print(val_losses)
    
    scheduler.step()
    
    print(f'epoch: {epoch:2}  Validation loss: {mean(val_losses)}')

MyModule(
  (layers): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)
epoch:  0  Training loss: 11.28199183409381
epoch:  0  Validation loss: 3.8106584161520005
epoch:  1  Training loss: 2.5805432280157783
epoch:  1  Validation loss: 2.4333407521247863
epoch:  2  Training loss: 1.9724630193345865
epoch:  2  Validation loss: 2.344310465455055
epoch:  3  Training loss: 1.9041940330699751
epoch:  3  Validation loss: 2.2653880298137663
epoch:  4  Training loss: 1.9099508022806446
epoch:  4  Validation loss: 2.2842288613319397
epoch:  5  Training loss: 1.878339552195968
epoch:  5  Validation loss: 2.4345241338014603
epoch:  6  Trainin

KeyboardInterrupt: 

In [33]:
#Training final model
model = MyModule().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001) #B1 = 0.9, β₂ = 0.999, ε = 1e-8.
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.01)
model.train()

for epoch in range(8):  
    train_losses = []
    for x_batch, y_batch in entire_training_loader: #entire training dataset
        x_batch, y_batch = x_batch.to(DEVICE),y_batch.to(DEVICE).unsqueeze(1)
        #print(x_batch.shape,y_batch.shape)
        y_pred = model(x_batch)
        #print(y_pred)
        loss = criterion(y_pred, y_batch)
        train_losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if(epoch%1 ==0):
        print(f'epoch: {epoch:2}  Training loss: {mean(train_losses)}')
        #print(train_losses)
    scheduler.step()
    
print("done")

epoch:  0  Training loss: 9.812678232484934
epoch:  1  Training loss: 2.43206479780528
epoch:  2  Training loss: 2.042910664057245
epoch:  3  Training loss: 1.9820669126145694
epoch:  4  Training loss: 1.899694744117406
epoch:  5  Training loss: 1.9046140246245327
epoch:  6  Training loss: 1.9556866872067353
epoch:  7  Training loss: 1.973508969861634
done


In [34]:
model.eval()
with torch.no_grad():
    results = []
    for [batch] in testing_loader:
        #print(type(batch))
        batch = batch.to(DEVICE)
        pred = model(batch).cpu().numpy()
        #manually clip predictions to lie within appropriate range
        pred[pred < 0] = 0.0 
        pred[pred > 10] = 10.0
        results.append(pred)
    results = np.vstack(results)
    print(results)
with open("result.txt", "w") as f:
    for val in np.concatenate(results):
        f.write(f"{val}\n")
print("Results have been saved")

[[ 2.7615716 ]
 [10.        ]
 [ 2.2099998 ]
 [ 5.1190567 ]
 [ 4.4733033 ]
 [ 6.582663  ]
 [ 5.030594  ]
 [ 4.802607  ]
 [ 4.3218813 ]
 [ 3.4906008 ]
 [ 9.462247  ]
 [ 3.8694098 ]
 [ 9.221952  ]
 [ 0.32143047]
 [ 1.2889714 ]
 [ 5.854158  ]
 [ 2.5678098 ]
 [ 7.985989  ]
 [ 7.9592433 ]
 [ 5.9201694 ]
 [ 5.1234984 ]
 [ 2.459309  ]
 [ 3.8693724 ]
 [ 5.8109927 ]
 [ 8.415081  ]
 [ 8.50062   ]
 [ 8.589104  ]
 [ 1.2367603 ]
 [ 4.6727543 ]
 [ 2.3214736 ]
 [ 5.234032  ]
 [ 2.525016  ]
 [ 7.088319  ]
 [ 5.4049597 ]
 [ 3.4108455 ]
 [ 7.2008576 ]
 [ 9.633412  ]
 [ 0.3042349 ]
 [ 6.578383  ]
 [ 2.3164496 ]
 [ 9.229493  ]
 [ 8.756834  ]
 [ 3.7283428 ]
 [ 3.0998693 ]
 [ 8.512824  ]
 [ 8.010607  ]
 [ 2.3053057 ]
 [ 5.9086194 ]
 [ 2.1999295 ]
 [ 2.1825504 ]
 [ 8.110398  ]
 [ 7.8188663 ]
 [ 9.79775   ]
 [ 2.4090223 ]
 [ 7.907233  ]
 [ 1.6220727 ]
 [ 6.8000436 ]
 [ 6.8418765 ]
 [ 3.3188944 ]
 [ 1.1474646 ]
 [ 7.2071886 ]
 [ 4.529659  ]
 [ 0.70803237]
 [ 7.156972  ]
 [ 5.5995407 ]
 [ 9.35598   ]
 [ 2.46414